# 15 — ONNX, inferência visual e modelos modernos

**EP · Data Science Knowledge Base | Análise e Classificação de Imagens | Notebook 15/16**

**Execução:** Reinicie o kernel e execute as células em ordem. Observe as instruções específicas e as dependências opcionais indicadas neste notebook.

**Pré-requisitos:** Python, NumPy e fundamentos de visualização; consulte os capítulos anteriores quando necessário.

**Escopo de validação:** Exemplos centrais com NumPy não demonstram inferência ONNX. ONNX Runtime e modelos externos são opcionais e requerem validação separada.


## Materiais complementares

- [Guia Técnico](./reference_materials/15_onnx_inferencia_moderna.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_onnx_inferencia_moderna.pdf)


## Como estudar este capítulo

1. Leia os fundamentos e consulte o Guia Técnico correspondente.
2. Reinicie o kernel e execute as células obrigatórias na sequência.
3. Confira tabelas, visualizações, saídas e limitações descritas no notebook.
4. Resolva o exercício antes de consultar a solução; execute laboratórios opcionais apenas em ambiente compatível.

**Pré-requisitos transversais:** Programação Python (01), visualização de dados (02), Estatística (03), Machine Learning (05) e Deep Learning (07), conforme o capítulo.


## Objetivos de aprendizagem

Distinguir treino, exportação e inferência, e validar metadados de entradas e saídas.

Ao concluir, você deverá ser capaz de **explicar a técnica**, **executar o exemplo**, **interpretar limitações** e **justificar escolhas de avaliação**.

## Fundamentos conceituais

ONNX é formato de intercâmbio de grafos computacionais, não arquitetura de rede. O pipeline de inferência exige shape (N,C,H,W) ou (N,H,W,C), normalização, ordem dos canais, pós-processamento e versões de operadores compatíveis. Comparar latência exige warmup, repetição, mesmo hardware e precisão numérica. Sem runtime instalado, validaremos a lógica com NumPy.

### Conceito matemático

$$latência\ média=\frac{1}{n}\sum_{i=1}^n t_i$$

A fórmula expressa o mecanismo principal do capítulo. Identifique unidades, dimensões e hipóteses antes de aplicar uma implementação.

## Experimento 1 — construção e medidas

Execute em kernel limpo. Exemplos usam datasets de bibliotecas instaláveis e não requerem download de arquivos externos.

In [1]:
import numpy as np,time
rng=np.random.default_rng(1)
img=rng.integers(0,256,(64,64,3),dtype=np.uint8)
x=img.astype('float32')/255.0
nchw=np.transpose(x,(2,0,1))[None]
assert nchw.shape==(1,3,64,64)
print('BGR/RGB precisa ser definido; entrada NCHW:',nchw.shape,'dtype:',nchw.dtype)

BGR/RGB precisa ser definido; entrada NCHW: (1, 3, 64, 64) dtype: float32


## Experimento 2 — visualização, comparação ou validação

Observe não apenas o valor obtido, mas o que o experimento **realmente demonstra**.

In [2]:
import numpy as np
# Demonstração de validação de saída; NÃO equivale a inferência ONNX.
logits=np.array([1.0,2.4,-0.2]);scores=np.exp(logits-logits.max());scores/=scores.sum()
print('Softmax sintético:',np.round(scores,3),'classe:',int(scores.argmax()))
assert np.isclose(scores.sum(),1)

Softmax sintético: [0.187 0.757 0.056] classe: 1


## Interpretação técnica

O exemplo implementa só preparo e pós-processamento; ONNX Runtime não é exigido e nenhum modelo foi carregado.

### Conexão com o acervo

Em vez de repetir capítulos de SVM, PCA, métricas e backpropagation, este notebook concentra a aplicação e limitações **em imagens**. Revisite as Disciplinas 05 e 07 para demonstrações matemáticas mais extensas.

## Exercício resolvido

**Problema:** Se o modelo espera RGB mas recebe BGR, a inferência é equivalente?

**Resolução comentada:** Não: a semântica dos canais muda e o desempenho pode degradar.

**Para praticar:** modifique um parâmetro do exemplo, registre a hipótese antes de executar e compare resultado, qualidade visual e custo computacional.

## Laboratório opcional — inferência ONNX Runtime em ambiente isolado

**Requer:** `~/onnx-env/bin/python` com `onnx`, `onnxruntime` e NumPy instalados. O kernel principal continua em `python3` (`jupyter-env`); a célula executa o teste no `onnx-env` por `subprocess`, sem trocar kernels nem instalar dependências.

Vamos **construir um grafo ONNX simples em memória**, carregar esse grafo no `CPUExecutionProvider` e executar a transformação aritmética `Y = 2 × X + 1` sobre o **tensor NCHW da imagem sintética** do primeiro experimento. O resultado será comparado com NumPy, com inspeção de metadados das entradas e saídas. Trata-se de inferência ONNX real em um grafo didático, **não de classificação visual ou avaliação de modelo pré-treinado**. Não há downloads nem arquivos versionados novos.

Se o ambiente não existir, a célula informa **NÃO EXECUTADO**; falhas de inferência serão exibidas como erro.

In [3]:
from pathlib import Path
import subprocess
import tempfile
import textwrap
import numpy as np

python_onnx = Path.home() / "onnx-env" / "bin" / "python"
if not python_onnx.is_file():
    print(f"LABORATÓRIO ONNX NÃO EXECUTADO: ambiente ausente ({python_onnx})")
else:
    # Tensor NCHW preparado anteriormente: (1, 3, 64, 64), RGB float32 normalizado.
    entrada_nchw = np.ascontiguousarray(nchw, dtype=np.float32)
    assert entrada_nchw.shape == (1, 3, 64, 64)
    with tempfile.TemporaryDirectory(prefix="ep08_onnx_") as pasta:
        arquivo = Path(pasta) / "entrada_nchw.npy"
        np.save(arquivo, entrada_nchw, allow_pickle=False)
        script = textwrap.dedent(r"""
            import sys
            import numpy as np
            import onnx
            import onnxruntime as ort
            from onnx import TensorProto, helper

            dado = np.load(sys.argv[1], allow_pickle=False)
            assert dado.shape == (1, 3, 64, 64) and dado.dtype == np.float32
            x = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 3, 64, 64])
            y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, 3, 64, 64])
            dois = helper.make_tensor("dois", TensorProto.FLOAT, [1], [2.0])
            um = helper.make_tensor("um", TensorProto.FLOAT, [1], [1.0])
            grafo = helper.make_graph([
                helper.make_node("Mul", ["X", "dois"], ["tmp"]),
                helper.make_node("Add", ["tmp", "um"], ["Y"]),
            ], "ep08_onnx_nchw", [x], [y], initializer=[dois, um])
            modelo = helper.make_model(
                grafo, opset_imports=[helper.make_operatorsetid("", 13)]
            )
            modelo.ir_version = 9
            onnx.checker.check_model(modelo)
            sessao = ort.InferenceSession(
                modelo.SerializeToString(), providers=["CPUExecutionProvider"]
            )
            saida = sessao.run(None, {"X": dado})[0]
            esperado = dado * 2.0 + 1.0
            assert saida.shape == dado.shape
            assert np.isfinite(saida).all()
            assert np.allclose(saida, esperado, rtol=1e-6, atol=1e-6)
            assert "CPUExecutionProvider" in sessao.get_providers()
            print("ONNX RUNTIME APROVADO — versão:", ort.__version__)
            print("Provider:", sessao.get_providers())
            print("Entrada:", [(i.name, i.shape, i.type) for i in sessao.get_inputs()])
            print("Saída:", [(o.name, o.shape, o.type) for o in sessao.get_outputs()])
            print("Erro absoluto máximo vs NumPy:", float(np.max(np.abs(saida - esperado))))
            print("Limitação: grafo aritmético; não há classificação de imagens.")
        """)
        try:
            teste = subprocess.run(
                [str(python_onnx), "-c", script, str(arquivo)],
                cwd=pasta, capture_output=True, text=True, timeout=120,
                check=False,
            )
        except subprocess.TimeoutExpired as erro:
            raise RuntimeError("O teste opcional ONNX excedeu 120 segundos.") from erro
        if teste.returncode:
            raise RuntimeError("Falha no ambiente onnx-env:\n" + teste.stderr)
        print(teste.stdout.rstrip())

ONNX RUNTIME APROVADO — versão: 1.32.0
Provider: ['CPUExecutionProvider']
Entrada: [('X', [1, 3, 64, 64], 'tensor(float)')]
Saída: [('Y', [1, 3, 64, 64], 'tensor(float)')]
Erro absoluto máximo vs NumPy: 0.0
Limitação: grafo aritmético; não há classificação de imagens.


## Erros comuns e limitações

Uma API de inferência precisa preservar exatamente o pré-processamento usado no treinamento.

- Nunca ajustar parâmetros em cima do conjunto de teste final.
- Não confundir demonstração com validação científica completa.
- Documentar versões, fontes de dados e condições de execução.
- Não redistribuir fotografias de rostos ou datasets históricos sem licença e autorização.

## Consulta rápida

- **Pergunta-chave:** Distinguir treino, exportação e inferência, e validar metadados de entradas e saídas.
- **Mecanismo:** ONNX é formato de intercâmbio de grafos computacionais, não arquitetura de rede.
- **Equação:** $latência\ média=\frac{1}{n}\sum_{i=1}^n t_i$
- **Verificação:** interpretar formato dos dados, hipótese e saída antes de utilizar em um projeto.

### Referências de estudo

- Apresentações originais da disciplina (PDF consolidado).
- Documentação: [scikit-image](https://scikit-image.org/docs/stable/), [OpenCV](https://docs.opencv.org/4.x/), [scikit-learn](https://scikit-learn.org/stable/), conforme o capítulo.
- Para CNNs: [Keras](https://keras.io/).

## Aprofundamento v2 — auditoria e aplicação

**Ponto de atenção deste capítulo:** dependência ONNX. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **dependência ONNX** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **dependência ONNX** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
